PART E

In [ ]:
import os
import ssl
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

In [ ]:
class BidirectionalHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)
        out = wei @ v
        return out

class MultiHeadBidirectionalAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([BidirectionalHead(n_embd, head_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class BlockBidirectional(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.sa = MultiHeadBidirectionalAttention(n_embd, n_head, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, num_classes=10, n_embd=192, n_head=6, n_layer=6, dropout=0.1):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_chans, n_embd, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, n_embd))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, n_embd))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([BlockBidirectional(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.norm = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, num_classes)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        cls_final = x[:, 0]
        return self.head(cls_final)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

os.makedirs('./data', exist_ok=True)

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2 if os.name == 'posix' else 0)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2 if os.name == 'posix' else 0)

c:\Users\vishw\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
model = ViT(img_size=32, patch_size=4, in_chans=3, num_classes=10, n_embd=64, n_head=4, n_layer=2, dropout=0.1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

vit_train_accuracies = []
vit_val_accuracies = []
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    batch_count = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
        
        batch_count += 1
        if batch_count >= 50:
            break
            
    train_acc = 100.0 * correct_train / total_train
    vit_train_accuracies.append(train_acc)

    model.eval()
    correct_val = 0
    total_val = 0
    val_batch_count = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
            
            val_batch_count += 1
            if val_batch_count >= 10:
                break
                
    val_acc = 100.0 * correct_val / total_val
    vit_val_accuracies.append(val_acc)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/batch_count:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

Epoch 1/3 | Loss: 2.1405 | Train Acc: 19.94% | Val Acc: 26.88%
Epoch 2/3 | Loss: 1.9947 | Train Acc: 25.27% | Val Acc: 27.66%
Epoch 3/3 | Loss: 1.9187 | Train Acc: 27.47% | Val Acc: 31.02%


In [ ]:
plt.figure(figsize=(10, 5))
# Use dynamic length based on actual epochs trained
epochs_trained = len(vit_train_accuracies)

plt.plot(range(1, epochs_trained + 1), vit_train_accuracies, label='ViT Train Accuracy', marker='o')
plt.plot(range(1, epochs_trained + 1), vit_val_accuracies, label='ViT Val Accuracy', marker='s')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Vision Transformer (ViT) Performance')
plt.legend()
plt.grid(True)
plt.savefig('vit_baseline_plot.png')
plt.close()

if os.path.exists("cnn_val_accs.pt"):
    cnn_accs = torch.load("cnn_val_accs.pt", map_location='cpu').tolist()
    plt.figure(figsize=(10, 5))
    
    # CNN trained for 10, ViT trained for 3. Dynamic ranges fix the crash.
    plt.plot(range(1, len(cnn_accs) + 1), cnn_accs, label='CNN Validation Accuracy (Part B)', color='red', linestyle='--', marker='o')
    plt.plot(range(1, len(vit_val_accuracies) + 1), vit_val_accuracies, label='ViT Validation Accuracy (Part E)', color='blue', linestyle='-', marker='s')
    
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('CIFAR-10 Validation Accuracy: CNN vs. ViT')
    plt.legend()
    plt.grid(True)
    plt.savefig('comparison_plot.png')
    plt.close()